# MindForge — Agent Playground

Manual scratchpad for playing with each agent as it lands. Not tests — just runnable cells.

**Setup:** copy `.env.example` → `.env`, fill in keys, then run Section 0 first.

---
**Sections**
- [0. Environment bootstrap](#0)
- [1. Knowledge Curator Agent](#1)
- [2. Curriculum Architect Agent](#2) ← add after Task 6.1
- [3. Teacher Agent](#3) ← add after Task 7.1
- [4. Interviewer Agent](#4) ← add after Task 8.1
- [5. Orchestrator Agent](#5) ← add after Task 9.1
- [6. Integration: chained cross-agent flow](#6) ← add after Task 9

## 0. Environment bootstrap <a id='0'></a>

Run once per kernel session. Adds repo root to `sys.path`, loads `.env`, configures Cognee.

In [ ]:
import sys, os, pprint

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from dotenv import load_dotenv
load_dotenv(os.path.join(REPO_ROOT, ".env"), override=True)

from mindforge.config import settings
print("LLM model  :", settings.llm_model)
print("Embedding  :", settings.embedding_model)
print("Mistral key:", "SET" if settings.mistral_api_key else "MISSING")

In [ ]:
import cognee

cognee.config.set_llm_config({
    "llm_provider": settings.llm_provider,
    "llm_model":    settings.llm_model,
    "llm_api_key":  settings.mistral_api_key,
    "llm_endpoint": settings.llm_endpoint,
})
cognee.config.set_embedding_config({
    "embedding_provider":   settings.embedding_provider,
    "embedding_model":      settings.embedding_model,
    "embedding_api_key":    settings.mistral_api_key,
    "embedding_dimensions": settings.embedding_dimensions,
    "embedding_endpoint":   settings.embedding_endpoint,
})
print("Cognee configured.")

## 1. Knowledge Curator Agent <a id='1'></a>

**What it does:** Ingests text/URLs, extracts concepts + prerequisite relationships via LLM, persists to Cognee permanent memory.

**Interacts with:**

| Direction | Cognee call | Scope |
|-----------|-------------|-------|
| Write | `cognee.remember()` | Permanent — dataset-scoped |
| Delete | `cognee.forget()` | Dataset or single item |

**Sample input:**
```python
content = "Backpropagation uses the chain rule..."
dataset = "playground_dl"
source_metadata = {"title": "Deep Learning Book", "author": "Goodfellow et al.", "year": 2016}
```

**Sample output:**
```
IngestionResult(concepts_count=3, relationships_count=2,
                concepts=['gradient_descent', 'backpropagation', 'chain_rule'],
                source_url='', dataset='playground_dl')
```

In [ ]:
from mindforge.agents.knowledge_curator import KnowledgeCuratorAgent

curator = KnowledgeCuratorAgent()

SAMPLE_TEXT = """
Gradient descent is an optimisation algorithm that iteratively adjusts model parameters
in the direction of the negative gradient of the loss function. It is the foundation
of neural network training.

Backpropagation computes gradients of the loss with respect to every weight by applying
the chain rule backwards through the computational graph. It requires gradient descent
as a prerequisite.

The chain rule states that the derivative of a composed function equals the product of
the derivatives of its components. It directly underlies backpropagation.
"""

result = await curator.ingest_content(
    content=SAMPLE_TEXT,
    dataset="playground_dl",
    source_metadata={
        "title":  "Deep Learning Book",
        "author": "Goodfellow, Bengio, Courville",
        "year":   2016,
        "url":    "https://www.deeplearningbook.org",
    },
)

print("=== IngestionResult ===")
pprint.pprint(result)

In [ ]:
# Ingest a live URL — fetches the page, then extracts concepts from it.
url_result = await curator.ingest_content(
    content="https://en.wikipedia.org/wiki/Backpropagation",
    dataset="playground_dl",
    source_metadata={
        "title": "Backpropagation — Wikipedia",
        "url":   "https://en.wikipedia.org/wiki/Backpropagation",
    },
)
print("=== URL IngestionResult ===")
pprint.pprint(url_result)

In [ ]:
# Inspect the resulting graph state in Cognee.
await cognee.visualize_graph(dataset="playground_dl")

In [ ]:
# remove_item: delete one specific item by ID.
await curator.remove_item("chain_rule")
print("chain_rule removed. Re-run visualize_graph to confirm.")

In [ ]:
# remove_topic: wipe the entire dataset.
# Comment out if you want data available for later sections.
await curator.remove_topic("playground_dl")
print("Dataset playground_dl removed.")

## 2. Curriculum Architect Agent <a id='2'></a>

> **Add cells here after Task 6.1 lands.**

**What it does:** Queries the concept graph, topologically sorts by prerequisites, returns a personalised learning path skipping mastered concepts.

**Interacts with:**

| Direction | Cognee call | Scope |
|-----------|-------------|-------|
| Read | `cognee.recall()` | Permanent — concept graph + learner profile |

**Sample input:**
```python
path = await architect.generate_learning_path(
    goal="deep learning", learner_id="alice", dataset="playground_dl"
)
```

**Sample output:**
```
LearningPath(goal='deep learning', total_concepts=4, estimated_hours=8.0,
             concepts=[ConceptStep('chain_rule'), ConceptStep('gradient_descent'),
                       ConceptStep('backpropagation'), ConceptStep('transformers')])
```

## 3. Teacher Agent <a id='3'></a>

> **Add cells here after Tasks 7.1–7.2 land.**

**What it does:** Runs a Socratic dialogue — presents a small explanation chunk, asks a probing question, evaluates the reply, and decides whether to advance to the next concept.

**Interacts with:**

| Direction | Cognee call | Scope |
|-----------|-------------|-------|
| Read | `cognee.recall()` | Permanent — concept definition |
| Read | `cognee.recall()` | Session — last 5 dialogue turns |
| Write | `cognee.remember()` | Session — each Q&A turn |

**Sample input:**
```python
resp = await teacher.teach_concept(
    concept_id="backpropagation", learner_id="alice",
    session_id="sess_001", dataset="playground_dl"
)
```

**Sample output:**
```
TeachingResponse(
    concept_id='backpropagation',
    explanation='Backpropagation computes gradients by applying the chain rule...',
    question='If a network has 3 layers, in what order does backprop compute gradients?',
    source='Deep Learning Book, Goodfellow et al. (2016)',
    turn=1
)
```

## 4. Interviewer Agent <a id='4'></a>

> **Add cells here after Tasks 8.1–8.2 land.**

**What it does:** Runs an adaptive quiz — retrieves weak concepts, generates 5 questions, adapts difficulty per answer, scores the session.

**Interacts with:**

| Direction | Cognee call | Scope |
|-----------|-------------|-------|
| Read | `cognee.recall()` | Permanent — weak concept lookup |
| Write | `cognee.remember()` | Session — each answer + correctness |
| Read | `cognee.recall()` | Session — all answers for final score |

**Sample input:**
```python
iv = await interviewer.start_interview(
    learner_id="alice", session_id="sess_002",
    dataset="playground_dl", num_questions=5
)
```

**Sample output:**
```
InterviewSession(session_id='sess_002', total_questions=5, current_question=1,
                 question_text='What is the role of the learning rate in gradient descent?',
                 concept_id='gradient_descent', difficulty='medium')
```

## 5. Orchestrator Agent <a id='5'></a>

> **Add cells here after Tasks 9.1–9.2 land.**

**What it does:** Unified entry point — creates sessions, routes intent to the right agent, calls `cognee.improve()` at session end to bridge session memory into the learner profile, handles resets via `cognee.forget()`.

**Interacts with:**

| Direction | Cognee call | Scope |
|-----------|-------------|-------|
| Read | `cognee.recall()` | Permanent — learner profile on session start |
| Write | `cognee.improve()` | Permanent — bridge session → learner profile |
| Delete | `cognee.forget()` | Learner profile reset or full wipe |

**Sample input:**
```python
session_id = await orchestrator.start_session(learner_id="alice")
resp = await orchestrator.route_request(
    intent="teach", learner_id="alice", session_id=session_id,
    payload={"concept_id": "backpropagation", "dataset": "playground_dl"}
)
await orchestrator.end_session(session_id, dataset="playground_dl")
```

## 6. Integration — chained cross-agent flow <a id='6'></a>

> **Activate when Orchestrator (Task 9) exists.**

Run cells in order to observe how memory written by one agent becomes visible to the next:

1. **Ingest** → KnowledgeCurator writes to permanent memory (`remember`)
2. **Learn** → CurriculumArchitect reads concept graph (`recall`), returns ordered path
3. **Teach ×3** → TeacherAgent reads concepts + session history, stores each turn (`recall` + `remember`)
4. **End session** → Orchestrator calls `improve()`, session memory bridges into learner profile
5. **Interview** → InterviewerAgent reads updated weak concepts (`recall`), 5 questions
6. **Finish** → `InterviewResults` + `visualize_graph()` showing updated feedback weights
7. **Reset** → Orchestrator calls `forget()`, profile wiped

In [ ]:
# Step 0: imports — uncomment after all agents are implemented
# from mindforge.agents.knowledge_curator    import KnowledgeCuratorAgent
# from mindforge.agents.curriculum_architect import CurriculumArchitectAgent
# from mindforge.agents.teacher              import TeacherAgent
# from mindforge.agents.interviewer          import InterviewerAgent
# from mindforge.agents.orchestrator         import OrchestratorAgent
#
# LEARNER_ID  = "playground_user"
# DATASET     = "integration_dl"
#
# curator      = KnowledgeCuratorAgent()
# architect    = CurriculumArchitectAgent()
# teacher      = TeacherAgent()
# interviewer  = InterviewerAgent()
# orchestrator = OrchestratorAgent()
print("Uncomment imports once all agents are implemented.")

In [ ]:
# Step 1 — Ingest (cognee.remember)
# ingest_result = await curator.ingest_content(
#     content="Gradient descent ... Backpropagation ... Transformers ...",
#     dataset=DATASET,
#     source_metadata={"title": "Integration Test Source", "year": 2024},
# )
# print("Ingested:", ingest_result)
print("Step 1 placeholder.")

In [ ]:
# Step 2 — Generate learning path (cognee.recall)
# path = await architect.generate_learning_path(
#     goal="deep learning", learner_id=LEARNER_ID, dataset=DATASET
# )
# print("Path:", [c.concept_id for c in path.concepts])
print("Step 2 placeholder.")

In [ ]:
# Step 3 — Teach 3 turns (cognee.recall + cognee.remember)
# session_id = await orchestrator.start_session(learner_id=LEARNER_ID)
#
# HARDCODED_ANSWERS = [
#     "From output to input using the chain rule.",
#     "Weights are adjusted proportionally to their gradient.",
#     "The learning rate controls the step size.",
# ]
#
# for i, step in enumerate(path.concepts[:3]):
#     t = await teacher.teach_concept(
#         concept_id=step.concept_id, learner_id=LEARNER_ID,
#         session_id=session_id, dataset=DATASET,
#     )
#     print(f"\n--- Turn {i+1}: {step.concept_id} ---")
#     print("Explanation:", t.explanation)
#     print("Question   :", t.question)
#     ev = await teacher.evaluate_response(
#         learner_response=HARDCODED_ANSWERS[i],
#         expected_concept=step.concept_id,
#         session_id=session_id,
#     )
#     print("Eval       :", ev.level, "| advance:", ev.advance)
print("Step 3 placeholder.")

In [ ]:
# Step 4 — End session → cognee.improve() bridges session → learner profile
# await orchestrator.end_session(session_id, dataset=DATASET)
# print(f"Session {session_id} ended. improve() called.")
print("Step 4 placeholder.")

In [ ]:
# Step 5 & 6 — Interview → finish (cognee.recall + cognee.remember)
# iv_session_id = await orchestrator.start_session(learner_id=LEARNER_ID)
# iv = await interviewer.start_interview(
#     learner_id=LEARNER_ID, session_id=iv_session_id,
#     dataset=DATASET, num_questions=5,
# )
# print("First question:", iv.question_text)
#
# # ... submit 4 more answers ...
#
# results = await interviewer.finish_interview(session_id=iv_session_id, dataset=DATASET)
# print("Score:", results.score, "| Weak concepts:", results.weak_concepts)
# await orchestrator.end_session(iv_session_id, dataset=DATASET)
print("Steps 5–6 placeholder.")

In [ ]:
# Inspect graph after full chain — feedback weights should reflect interview results.
# await cognee.visualize_graph(dataset=DATASET)
print("Graph inspect placeholder — uncomment after Steps 1–6.")

In [ ]:
# Step 7 — Reset (cognee.forget)
# await orchestrator.reset_learner(LEARNER_ID)  # removes only this learner's profile
# await orchestrator.reset_all()                # nuclear — wipes everything
print("Step 7 placeholder.")